# LangChain Concepts Cookbook with OCI

Chat, tools, agents, structured output, and embeddings with synthetic examples. See [readme.md](readme.md) for setup and details.


In [1]:
# ==============================================================================
# Scenario: Notebook dependency setup
# Context: Install the libraries used by the LangChain examples.
# ==============================================================================

%pip install "langchain==1.4.5" "langchain-oci==0.3.2" "python-dotenv>=1,<2" "numpy>=1.24,<3" "pydantic>=2.7,<3"


Note: you may need to restart the kernel to use updated packages.


## 1. Chat models: configuration and initialization

Initialize `ChatOCIGenAI` using settings from `.env`.


In [2]:
# ==============================================================================
# Scenario: OCI chat model setup
# Context: Load local configuration and initialize the shared chat model.
# ==============================================================================

import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_oci import ChatOCIGenAI

project_root = Path.cwd()
if not (project_root / "oci-langchain-cookbook.ipynb").is_file():
    raise RuntimeError("Start Jupyter from the folder containing this notebook.")
load_dotenv(project_root / ".env")

compartment_id = os.getenv("OCI_COMPARTMENT_ID", "").strip()
service_endpoint = os.getenv("OCI_SERVICE_ENDPOINT", "").strip()
chat_model_id = "openai.gpt-oss-120b"
required_settings = {
    "OCI_COMPARTMENT_ID": compartment_id,
    "OCI_SERVICE_ENDPOINT": service_endpoint,
}
missing = [name for name, value in required_settings.items() if not value]
if missing:
    raise ValueError("Set these values in .env: " + ", ".join(missing))

oci_settings = {
    "compartment_id": compartment_id,
    "service_endpoint": service_endpoint,
    "auth_type": "API_KEY",
    "auth_profile": os.getenv("OCI_CONFIG_PROFILE", "").strip() or "DEFAULT",
    "auth_file_location": str(Path(
        os.getenv("OCI_CONFIG_FILE", "").strip() or "~/.oci/config"
    ).expanduser()),
}
llm = ChatOCIGenAI(model_id=chat_model_id, **oci_settings)
print("OCI chat initialized; configuration values are not displayed.")


OCI chat initialized; configuration values are not displayed.


## 2. Chat messages and conversation history

Pass system, human, and AI messages to `invoke()`.


In [3]:
# ==============================================================================
# Scenario: Hospitality guest-services conversation
# Context: Respond to a guest using supplied hotel policy and conversation history.
# ==============================================================================

system_prompt = (
    "You are a helpful and polite guest-services assistant for the Fictional Harbor Hotel. "
    "Adhere strictly to the following policy: "
    "1. Standard check-in begins at 15:00. "
    "2. Early check-in is not guaranteed and depends entirely on room availability. "
    "3. Complimentary luggage storage is always available for guests. "
    "Never promise early check-in availability. Keep replies concise."
)
# The conversation is now a series of turns between the user (guest) and the assistant.
conversation_messages = [
    ("system", system_prompt),
    
    # --- Turn 1 ---
    ("human", "Hi, I have a reservation and I'd like to check in. I know I'm quite early."),
    # Expected AI response based on the system prompt
    ("ai", "Welcome to the Harbor Hotel! Our standard check-in time begins at 15:00. "
           "I can check if your room is ready, but I can't guarantee an early check-in at this time."),

    # --- Turn 2 ---
    ("human", "Ah, okay. What should I do in the meantime? My bags are really heavy."),
    # Expected AI response based on the system prompt
    ("ai", "We would be happy to store your luggage for you free of charge. You can then explore "
           "the harbor area and we will contact you if your room becomes available any sooner."),
    
    # --- Turn 3 ---
    ("human", "That would be perfect, thank you!"),
    # Expected AI response
    ("ai", "You're very welcome! We'll take care of your bags right here. Enjoy your morning!")
]

bot_response = llm.invoke(conversation_messages)
print(bot_response.content)


My pleasure! If you need anything else, just let us know. Enjoy your stay!


## 3. Streaming chat responses

Read response chunks with `stream()`.


In [4]:
# ==============================================================================
# Scenario: Government citizen-service response
# Context: Stream an explanation of a fictional municipal application process.
# ==============================================================================

system_prompt = (
    "You are a friendly and helpful assistant for the 'City of Evergreen's Green Energy Program'. "
    "Your role is to explain the program's process to residents in simple, encouraging, plain language. "
    "Adhere strictly to the following process description: "
    
    "1.  **Eligibility Check:** The program is for single-family homeowners within city limits who are current on their property taxes. "
    "2.  **Application Packet:** To apply, residents must submit three items: the completed Application Form (G-1), a recent utility bill as proof of residence, and a project quote from a certified solar installer. "
    "3.  **Review Process:** After submission, the packet goes through a three-stage review. "
    "    - First, an administrative review checks if all documents are present. Incomplete packets are returned with instructions. "
    "    - Second, a technical review team assesses the installer's quote for compliance with program standards. "
    "    - Third, the Grant Committee conducts a final review and makes the decision. "
    
    "**Your constraints:** "
    "- Do NOT discuss specific grant amounts, dollar figures, or deadlines. "
    "- Do NOT guarantee approval or provide a timeline for a decision. "
    "- Do NOT offer legal or financial advice. "
    "- Your goal is to explain the *process* clearly and calmly."
)


# The conversation unfolds in a series of natural user questions and AI responses.
conversation_messages = [
    ("system", system_prompt),

    # --- Turn 1 ---
    ("human", "Hi, I heard the city has a program to help with solar panels. How does it work?"),
    # Expected AI Response: A brief, welcoming overview.
    ("ai", "Welcome! Yes, our Green Energy Program is designed to help homeowners like you. "
           "The process involves an eligibility check, submitting an application packet, and then a multi-stage review. "
           "Would you like me to walk you through the details?"),

    # --- Turn 2 (This turn is designed to elicit the lengthy, streamable response) ---
    ("human", "Yes please! I think I'm probably eligible. Can you explain the application and review process in detail?"),
]

for chunk in llm.stream(conversation_messages):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()


### 1. Eligibility Check  
Before you start the paperwork, make sure you meet the two basic requirements:

1. **You own a single‑family home** that is located inside the City of Evergreen limits.  
2. **Your property taxes are current** – meaning there are no past‑due taxes on the account.

If both of those are true, you’re eligible to apply.

---

### 2. Application Packet – What You Need to Send

To begin the process you’ll prepare a single packet that includes three items:

| Item | Why we need it | How to obtain it |
|------|----------------|------------------|
| **Completed Application Form (G‑1)** | Shows your interest and provides basic information about you and your home. | Download the form from the City’s website or pick one up at the Planning & Development Office. Fill it out completely and sign it. |
| **Recent utility bill** | Serves as proof that the address belongs to you. | A bill from the last 30‑60 days (electric, gas, water, etc.) that clearly shows your name and the

## 4. Async invocation and streaming

Use `ainvoke()` concurrently and consume `astream()`.


In [5]:
# ==============================================================================
# Scenario: Financial-operations ticket handoff
# Context: Summarize synthetic tickets concurrently and stream a staff handoff.
# ==============================================================================

import asyncio

operations_tickets = [
    (
        "PROD-LL-789: The nightly auto-loan portfolio reconciliation job (`recon-prod-nightly-01`) "
        "is critically delayed. The job, normally complete by 03:00 UTC, stalled at 02:15 UTC "
        "during the 'Cross-Portfolio Validation' step. Initial log analysis points to potential "
        "database contention; the DBA team is investigating for locks. "
        "Impact: The T+0 risk exposure dashboard will be unavailable for the trading desk at market open. "
        "No client-facing services are currently affected."
    ),

    (
        "PROD-LL-790: The live Lease Origination & Performance Dashboard is unavailable for all users. "
        "The front-end is timing out with a 'Cannot Fetch Data' error, and the backing API for "
        "lease valuation (`lease-val-api-v2`) is returning intermittent HTTP 503 service unavailable errors. "
        "Workaround: Staff have been instructed to use the documented 'Manual Valuation Tool' via Citrix as a "
        "temporary fallback for critical lease-end calculations. The application support team has "
        "escalated to DevOps for microservice health diagnostics."
    ),
]
ticket_summaries = await asyncio.gather(*[
    llm.ainvoke("Summarize this synthetic internal ticket in one sentence. "
                "Do not infer causes, losses, or deadlines.\n" + ticket)
    for ticket in operations_tickets
])
for ticket, summary in zip(operations_tickets, ticket_summaries):
    print(f"{ticket.split(':', 1)[0]}: {summary.content}")

async for chunk in llm.astream(
    "Draft a two-sentence internal handoff from these fictional tickets. "
    "Use only the supplied facts.\n" + "\n".join(operations_tickets)
):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()
await llm.aclose()


PROD-LL-789: The nightly auto‑loan portfolio reconciliation job (`recon-prod-nightly-01`) stalled during the “Cross‑Portfolio Validation” step, delaying its completion and rendering the T+0 risk‑exposure dashboard unavailable for the trading desk at market open.
PROD-LL-790: The Lease Origination & Performance Dashboard is unavailable for all users due to front‑end timeouts and intermittent 503 errors from the lease‑val‑api‑v2 service, and staff are being directed to use the Manual Valuation Tool via Citrix while support escalates the issue to DevOps.
PROD‑LL‑789: The nightly auto‑loan portfolio reconciliation job (recon‑prod‑nightly‑01) is critically delayed, having stalled at 02:15 UTC during the “Cross‑Portfolio Validation” step, and the T+0 risk exposure dashboard will be unavailable for the trading desk at market open while the DBA team investigates possible database contention; no client‑facing services are currently affected.  

PROD‑LL‑790: The live Lease Origination & Performa

## 5. Tools: definition, binding, and invocation

Define tools with `@tool`, bind their schemas, and execute selected calls.


In [6]:
# ==============================================================================
# Scenario: Hospitality reservation and service lookups
# Context: Select and execute read-only tools against synthetic hotel records.
# ==============================================================================

import os
from typing import Dict, Any
from langchain_core.tools import tool

# 1. Synthetic Backend Data Store (Simulating Hospitality PMS & Service Desk)
reservation_records: Dict[str, Dict[str, Any]] = {
    "RES-DEMO-101": {
        "guest_name": "Eleanor Vance",
        "status": "confirmed",
        "room_type": "standard_king",
        "nights": 2,
        "check_in": "2026-10-15",
    },
    "RES-DEMO-102": {
        "guest_name": "Arthur Pendelton",
        "status": "checked_in",
        "room_type": "deluxe_suite",
        "nights": 4,
        "check_in": "2026-10-14",
    },
}

service_records: Dict[str, Dict[str, Any]] = {
    "SR-DEMO-204": {
        "guest_name": "Eleanor Vance",
        "status": "assigned",
        "request": "extra towels and feather pillows",
        "target_room": "304",
    },
    "SR-DEMO-205": {
        "guest_name": "Arthur Pendelton",
        "status": "in_progress",
        "request": "late checkout inquiry",
        "target_room": "512",
    },
}

# 2. Tool Definitions with Explicit Schemas & Docstrings
@tool
def get_reservation_status(reservation_id: str) -> dict:
    """Look up a synthetic hotel reservation record using its exact alphanumeric reservation ID (e.g., 'RES-DEMO-101')."""
    record = reservation_records.get(reservation_id)
    if record is not None:
        return {"reservation_id": reservation_id, "found": True, **record}
    return {"reservation_id": reservation_id, "found": False, "status": "not_found"}

@tool
def get_guest_service_status(request_id: str) -> dict:
    """Look up the status and details of an existing guest-service ticket using its request ID (e.g., 'SR-DEMO-204')."""
    record = service_records.get(request_id)
    if record is not None:
        return {"request_id": request_id, "found": True, **record}
    return {"request_id": request_id, "found": False, "status": "not_found"}

# 3. Model Binding and Tool Invocation
hospitality_tools = [get_reservation_status, get_guest_service_status]
tool_map = {t.name: t for t in hospitality_tools}

llm_with_tools = llm.bind_tools(hospitality_tools)

# Simulation: Front-desk query to check a guest reservation
user_prompt = "check the details for reservation RES-DEMO-101?"
tool_selection = llm_with_tools.invoke(user_prompt)

# 4. Tool Execution & Output Inspection
if tool_selection.tool_calls:
    print(f"Agent Prompt: {user_prompt}\n")
    for call in tool_selection.tool_calls:
        tool_name = call["name"]
        tool_args = call["args"]
        print(f"-> Selected Tool: {tool_name}")
        print(f"-> Arguments:     {tool_args}")
        
        # Execute the tool call against the synthetic backend
        if tool_name in tool_map:
            result = tool_map[tool_name].invoke(tool_args)
            print(f"-> Result:        {result}\n")
else:
    print("No tool selected; check the model's tool-calling support or prompt phrasing.")

Agent Prompt: check the details for reservation RES-DEMO-101?

-> Selected Tool: get_reservation_status
-> Arguments:     {'reservation_id': 'RES-DEMO-101'}
-> Result:        {'reservation_id': 'RES-DEMO-101', 'found': True, 'guest_name': 'Eleanor Vance', 'status': 'confirmed', 'room_type': 'standard_king', 'nights': 2, 'check_in': '2026-10-15'}



/Users/sakthivelthangaraj/stepstogoogle/langchain/.venv/lib/python3.14/site-packages/langchain_oci/chat_models/oci_generative_ai.py:510: UserWarning: GenericProvider could not extract text and returned an empty string. Ensure the selected provider matches the response payload format, otherwise content extraction will return an empty string.
  content = self._provider.chat_response_to_text(response)


## 6. Tool calling: a bounded execution loop

Return tool results with `ToolMessage` and preserve call IDs.


In [7]:
# ==============================================================================
# Scenario: Hospitality concierge tool loop
# Context: Retrieve reservation and service updates through a bounded tool loop.
# ==============================================================================

import json
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# 1. Establish Tool Registry and System Guardrails
tool_registry = {item.name: item for item in hospitality_tools}

concierge_system_prompt = (
    "You are an efficient, polite hotel concierge assistant. "
    "Use the available lookup tools to check reservation and guest service statuses. "
    "Never hallucinate or guess IDs, room numbers, or service details. "
    "If IDs are missing, ask for clarification. Once all tool data is retrieved, "
    "synthesize the findings into a clear, professional summary for the guest."
)

conversation_history = [
    SystemMessage(content=concierge_system_prompt),
    HumanMessage(
        content="Check reservation RES-DEMO-101 and service request SR-DEMO-204 and give me a consolidated status update on both?"
    ),
]

# 2. Bounded ReAct / Tool-Calling Loop (Max 5 turns to prevent infinite cycles)
max_iterations = 5
print("=== Starting Autonomous Concierge Execution Loop ===\n")

for turn in range(1, max_iterations + 1):
    print(f"--- Iteration {turn} ---")
    response = llm_with_tools.invoke(conversation_history)
    conversation_history.append(response)

    # Termination Condition: No further tool calls requested -> Output final response
    if not response.tool_calls:
        print("\n=== Concierge Final Response to Guest ===")
        print(response.content)
        break

    # Process all tool calls requested by the model in this turn
    print(f"Model requested {len(response.tool_calls)} tool call(s):")
    for call in response.tool_calls:
        tool_name = call["name"]
        tool_args = call["args"]
        call_id = call["id"]

        print(f"  -> Calling '{tool_name}' with args {tool_args} [ID: {call_id}]")

        # Security check: Ensure requested tool is explicitly allowlisted
        if tool_name not in tool_registry:
            raise ValueError(f"Security Alert: Model attempted to call unsupported tool '{tool_name}'.")

        # Deterministic execution
        tool_output = tool_registry[tool_name].invoke(tool_args)
        print(f"     Result: {tool_output}")

        # Append structured ToolMessage result back to conversation context
        conversation_history.append(
            ToolMessage(
                content=json.dumps(tool_output),
                tool_call_id=call_id,
                name=tool_name,
            )
        )
    print()
else:
    raise RuntimeError(f"Tool execution limit ({max_iterations} turns) reached without convergence.")

=== Starting Autonomous Concierge Execution Loop ===

--- Iteration 1 ---


/Users/sakthivelthangaraj/stepstogoogle/langchain/.venv/lib/python3.14/site-packages/langchain_oci/chat_models/oci_generative_ai.py:510: UserWarning: GenericProvider could not extract text and returned an empty string. Ensure the selected provider matches the response payload format, otherwise content extraction will return an empty string.
  content = self._provider.chat_response_to_text(response)


Model requested 1 tool call(s):
  -> Calling 'get_reservation_status' with args {'reservation_id': 'RES-DEMO-101'} [ID: chatcmpl-tool-92c1ac0e5c3ee28f]
     Result: {'reservation_id': 'RES-DEMO-101', 'found': True, 'guest_name': 'Eleanor Vance', 'status': 'confirmed', 'room_type': 'standard_king', 'nights': 2, 'check_in': '2026-10-15'}

--- Iteration 2 ---
Model requested 1 tool call(s):
  -> Calling 'get_guest_service_status' with args {'request_id': 'SR-DEMO-202'} [ID: chatcmpl-tool-b50232fbe48dc7e5]
     Result: {'request_id': 'SR-DEMO-202', 'found': False, 'status': 'not_found'}

--- Iteration 3 ---


/Users/sakthivelthangaraj/stepstogoogle/langchain/.venv/lib/python3.14/site-packages/langchain_oci/chat_models/oci_generative_ai.py:510: UserWarning: GenericProvider could not extract text and returned an empty string. Ensure the selected provider matches the response payload format, otherwise content extraction will return an empty string.
  content = self._provider.chat_response_to_text(response)


Model requested 1 tool call(s):
  -> Calling 'get_guest_service_status' with args {'request_id': 'SR-DEMO-204'} [ID: chatcmpl-tool-b37a104f9b7cc39d]
     Result: {'request_id': 'SR-DEMO-204', 'found': True, 'guest_name': 'Eleanor Vance', 'status': 'assigned', 'request': 'extra towels and feather pillows', 'target_room': '304'}

--- Iteration 4 ---

=== Concierge Final Response to Guest ===
**Guest: Eleanor Vance**

| Item | Details | Current Status |
|------|---------|----------------|
| **Reservation (RES‑DEMO‑101)** | • Room type: Standard King<br>• Stay: 2 nights<br>• Check‑in date: 15 Oct 2026 (checkout 17 Oct 2026) | **Confirmed** – the reservation is secured and ready for arrival. |
| **Service Request (SR‑DEMO‑204)** | • Request: Extra towels and feather‑down pillows<br>• Assigned to room 304 | **Assigned** – the housekeeping team has been notified and will provide the requested items to room 304 shortly. |

**What to Expect**

- Your room will be ready on 15 Oct 2026. If you ne

## 7. Agents with create_agent

Let `create_agent` manage the tool loop and message history.


In [8]:
# ==============================================================================
# Scenario: Hospitality concierge agent
# Context: Run a tool-enabled agent and verify that both lookup tools executed.
# ==============================================================================

from typing import Set
from langchain_core.messages import ToolMessage
from langchain.agents import create_agent

# 1. System Guardrails & Prompt Construction
system_prompt = (
    "You are a professional hotel front-desk concierge assistant. "
    "Your primary responsibility is to assist hotel staff and guests by retrieving "
    "verified, synthetic database records for reservation and guest-service tickets.\n\n"
    "Operational Rules:\n"
    "1. Always use the provided tools ('get_reservation_status' and 'get_guest_service_status') "
    "for checking records. Never invent IDs, availability, guest names, or service statuses.\n"
    "2. If an ID is missing, explicitly prompt the user for clarification.\n"
    "3. Once tool results are retrieved, synthesize them into a concise, professional summary."
)

# 2. Agent Instantiation
hospitality_agent = create_agent(
    model=llm,
    tools=hospitality_tools,
    system_prompt=system_prompt,
)

# 3. Execution with Bound Recursion Limit
user_query = (
    "Look up reservation RES-DEMO-101 and guest-service request SR-DEMO-204 and provide a summary of both statuses for the front-desk logs."
)

agent_result = hospitality_agent.invoke(
    {"messages": [{"role": "user", "content": user_query}]},
    config={"recursion_limit": 12},
)

# 4. Telemetry and Tool Execution Audit
executed_tool_messages = [
    message for message in agent_result["messages"]
    if isinstance(message, ToolMessage)
]
executed_tool_names: Set[str] = {message.name for message in executed_tool_messages}
required_tools: Set[str] = {"get_reservation_status", "get_guest_service_status"}

print("=== Execution Trace & Tool Audit ===")
print(f"User Query: {user_query}")
print(f"Tools Executed: {list(executed_tool_names)}")

# Enforce deterministic verification that both tools were invoked
assert required_tools.issubset(executed_tool_names), (
    f"Audit Failure: Expected tools {required_tools} to execute, but got {executed_tool_names}. "
    "Inspect model tool-calling trajectories and prompts."
)

print("\n=== Concierge Staff Summary ===")
final_response = agent_result["messages"][-1].content
print(final_response)
print("\nAgent verification passed: Both required lookup tools were successfully invoked.")

/Users/sakthivelthangaraj/stepstogoogle/langchain/.venv/lib/python3.14/site-packages/langchain_oci/chat_models/oci_generative_ai.py:510: UserWarning: GenericProvider could not extract text and returned an empty string. Ensure the selected provider matches the response payload format, otherwise content extraction will return an empty string.
  content = self._provider.chat_response_to_text(response)


=== Execution Trace & Tool Audit ===
User Query: Look up reservation RES-DEMO-101 and guest-service request SR-DEMO-204 and provide a summary of both statuses for the front-desk logs.
Tools Executed: ['get_reservation_status', 'get_guest_service_status']

=== Concierge Staff Summary ===
**Reservation Summary (RES-DEMO-101)**
- **Guest:** Eleanor Vance  
- **Status:** Confirmed  
- **Room Type:** Standard King  
- **Nights:** 2  
- **Check‑in Date:** 15 Oct 2026  

**Guest‑Service Request Summary (SR-DEMO-204)**
I’m unable to locate the request SR‑DEMO‑204 in the system. Could you please verify the request ID or provide any additional details? This will allow me to retrieve the correct record for the front‑desk log.

Agent verification passed: Both required lookup tools were successfully invoked.


## 8. Structured output with Pydantic

Extract validated fields with `with_structured_output()`.


In [9]:
# ==============================================================================
# Scenario: Government service-incident extraction
# Context: Extract a validated incident record from a synthetic operations report.
# ==============================================================================

from typing import Literal
from pydantic import BaseModel, Field

class ServiceIncident(BaseModel):
    """Fields extracted from a fictional municipal operations report."""
    incident_id: str = Field(description="Exact incident ID in the report")
    priority: Literal["low", "medium", "high", "critical"]
    service: str
    summary: str
    affected_channels: list[str]
    requires_staff_review: bool

structured_llm = llm.with_structured_output(ServiceIncident)
incident = structured_llm.invoke(
    "Extract only the supplied facts: Fictional incident GOV-DEMO-301 has "
    "high priority. The municipal appointment booking service is unavailable "
    "on the web portal and lobby kiosks. Telephone booking remains available. "
    "Staff review is required before publishing any public notice."
)
print(incident.model_dump_json(indent=2))


{
  "incident_id": "GOV-DEMO-301",
  "priority": "high",
  "service": "municipal appointment booking",
  "summary": "Service is unavailable on the web portal and lobby kiosks; telephone booking remains available.",
  "affected_channels": [
    "web portal",
    "lobby kiosks"
  ],
  "requires_staff_review": true
}


/Users/sakthivelthangaraj/stepstogoogle/langchain/.venv/lib/python3.14/site-packages/langchain_oci/chat_models/oci_generative_ai.py:510: UserWarning: GenericProvider could not extract text and returned an empty string. Ensure the selected provider matches the response payload format, otherwise content extraction will return an empty string.
  content = self._provider.chat_response_to_text(response)


## 9. Multimodal chat: image input (optional)

Send text and a local image to a vision-capable chat model.


In [10]:
# ==============================================================================
# Scenario: Government architecture diagram inspection
# Context: Describe a local diagram with a separate vision-capable chat model.
# ==============================================================================

from langchain_oci import load_image

vision_model_id = "openai.gpt-5.6-luna"
vision_image_path = "sample_design_diagram.png"

image_path = Path(vision_image_path).expanduser()
if not image_path.is_absolute():
    image_path = project_root / image_path
if not image_path.is_file():
    raise FileNotFoundError("The configured vision image does not exist.")
vision_llm = ChatOCIGenAI(model_id=vision_model_id, **oci_settings)
vision_response = vision_llm.invoke([HumanMessage(content=[
    {"type": "text", "text": "Describe the visible components and labeled "
     "connections in this government service architecture diagram. "
     "Report unreadable labels and do not infer security controls."},
    load_image(str(image_path)),
])])
print(vision_response.content)


The diagram is titled **“Generic Service Portal — System Design”** and shows these components:

- **User**
- **Web Portal**
- **Backend API**
- **Database**
- **Notification Service**

Labeled connections:

- **User → Web Portal:** “Submit request”
- **Web Portal → Backend API:** “HTTPS”
- **Backend API → Database:** “Save and retrieve”
- **Backend API → Notification Service:** “Send confirmation”

The footer reads **“Fictional educational example.”** No labels appear unreadable.


## 10. Text embeddings and semantic search (optional)

Embed documents and a query, then rank by cosine similarity.


In [11]:
# ==============================================================================
# Scenario: Financial-operations procedure search
# Context: Retrieve a fictional procedure by comparing document and query embeddings.
# ==============================================================================

import numpy as np
from langchain_oci import OCIGenAIEmbeddings

procedure_documents = [
    # SOP 1: Clearing, Settlement & Exceptions
    """SOP-FIN-101: End-of-Day Settlement Discrepancy & Reconciliation
    
    Category: Core Clearing & Settlement Operations
    Scope: Daily settlement batch processing for ACH, Wire, and Card rails.
    
    Trigger:
    - Daily settlement reconciliation job flags unbalanced positions between 
      the internal core ledger and external clearinghouse/correspondent bank settlement reports.
    
    Procedure:
    1. Ingestion & Comparison: Extract the daily settlement feed from the clearinghouse portal 
       and compare transaction IDs against the internal general ledger posting journal.
    2. Discrepancy Identification: Identify unmatched debits/credits, timing discrepancies, 
       or clearing fee variances exceeding the $0.00 tolerance threshold.
    3. Queue Routing: Immediately route any unmatched line items or broken trades to the 
       Operations Exception Queue with tag 'SETTLE-DISCREPANCY'.
    4. Escalation: If unmatched volume exceeds $50,000 or remains open past 17:00 EST, 
       escalate directly to the Cash Management Supervisor for manual journal intervention.
    
    Audit & Compliance: Retain raw clearing files and reconciliation sign-off sheets for 7 years.""",

    # SOP 2: System Resilience & Operational Continuity
    """SOP-OPS-204: Retail Branch Dashboard Outage & Fallback Procedures
    
    Category: Infrastructure Continuity & Teller Operations
    Scope: Front-office branch systems, teller dashboards, and cash counter applications.
    
    Trigger:
    - Complete or intermittent loss of connectivity to the central Teller Portal or Branch Dashboard.
    
    Procedure:
    1. Incident Verification: Confirm outage severity with regional lead and verify if local 
       caching or network gateway failure is the root cause.
    2. Fallback Activation: Switch branch workstations to the offline manual reporting spreadsheet 
       (Template SEC-FALLBACK-v4) stored on the local branch encrypted network drive.
    3. Shift Handoff Logging: Record the exact system downtime, ticket ID from IT Service Desk, 
       and affected teller terminal numbers into the electronic Shift Handoff Log.
    4. Post-Restoration Catch-up: Once connectivity is restored, execute dual-sign-off batch 
       re-entry of manual transactions within 2 hours of system availability.
    
    Audit & Compliance: All offline transaction logs require co-signature by the Branch Manager.""",

    # SOP 3: Document Operations & Customer Communications
    """SOP-DOC-312: Duplicate Customer Statement Delivery & Suppression Remediation
    
    Category: Document Operations & Data Privacy
    Scope: Monthly account statement generation, e-delivery dispatch, and print-mail fulfillment.
    
    Trigger:
    - Automated delivery monitor detects duplicate statement distribution jobs or customer-reported 
      receipt of multiple statement copies for a single billing cycle.
    
    Procedure:
    1. Dispatch Verification: Cross-reference the distribution run ID against the central 
       Delivery Audit Log to determine if the issue is batch duplication or web portal re-triggering.
    2. Delivery Suppression: Immediately suspend downstream print queues and pause outgoing 
       e-delivery notification webhooks for the impacted batch sequence.
    3. Handoff & Remediation: Route confirmed duplicate instances and affected Account UUIDs 
       directly to the Document Operations Resolution Team.
    4. Privacy Assessment: Verify whether duplicate mailings exposed personally identifiable 
       information (PII) to unauthorized parties; notify Compliance if cross-mailing occurred.
    
    Audit & Compliance: Retain postal manifest logs and batch suppression receipts for audit review.""",

    # SOP 4: Fraud Detection & Account Restrictions
    """SOP-FRD-408: High-Velocity Wire Transfer Fraud Screening
    
    Category: Financial Crime & Fraud Operations
    Scope: Outbound wire transfers exceeding customer velocity profiles or flagged by risk rules.
    
    Trigger:
    - Real-time AML/Fraud monitoring engine triggers Alert Code AML-770 (Unusual Wire Velocity).
    
    Procedure:
    1. Temporary Hold: Automatically place an administrative hold on the pending wire instruction.
    2. Customer Callback Verification: Perform an out-of-band security callback using the primary 
       verified phone number on file; do not accept inbound voice confirmations.
    3. Beneficiary Validation: Verify the recipient routing and account number against known 
       mule account databases and international sanctions lists.
    4. Disposition: If fraud is suspected, cancel transaction, freeze online access, and file 
       a SAR (Suspicious Activity Report) referral with the Financial Crime Unit within 24 hours.
    
    Audit & Compliance: Document all customer voice logs and risk scoring metrics in the case file.""",
]

embedding_model_id = "openai.text-embedding-3-small"

embeddings = OCIGenAIEmbeddings(model_id=embedding_model_id, **oci_settings)
# Generate dense vector representations
document_vectors = np.asarray(embeddings.embed_documents(procedure_documents))
query_vector = np.asarray(
    embeddings.embed_query(
        "Which procedure covers entries that do not match the settlement report?"
    )
)

# Compute cosine similarities and rank procedures
denominators = np.linalg.norm(document_vectors, axis=1) * np.linalg.norm(query_vector)
if np.any(denominators == 0):
    raise ValueError("Cannot rank zero-length embedding vectors.")

similarities = (document_vectors @ query_vector) / denominators
best_index = int(np.argmax(similarities))

print(f"Top Matched Procedure:\n{procedure_documents[best_index]}\n")
print(f"Cosine similarity: {similarities[best_index]:.3f}")

Top Matched Procedure:
SOP-FIN-101: End-of-Day Settlement Discrepancy & Reconciliation

    Category: Core Clearing & Settlement Operations
    Scope: Daily settlement batch processing for ACH, Wire, and Card rails.

    Trigger:
    - Daily settlement reconciliation job flags unbalanced positions between 
      the internal core ledger and external clearinghouse/correspondent bank settlement reports.

    Procedure:
    1. Ingestion & Comparison: Extract the daily settlement feed from the clearinghouse portal 
       and compare transaction IDs against the internal general ledger posting journal.
    2. Discrepancy Identification: Identify unmatched debits/credits, timing discrepancies, 
       or clearing fee variances exceeding the $0.00 tolerance threshold.
    3. Queue Routing: Immediately route any unmatched line items or broken trades to the 
       Operations Exception Queue with tag 'SETTLE-DISCREPANCY'.
    4. Escalation: If unmatched volume exceeds $50,000 or remains open pa

## 11. Multimodal embeddings: text-to-image search (optional)

Rank diagrams using a shared text-image embedding space.


In [12]:
# ==============================================================================
# Scenario: Government architecture diagram search
# Context: Find a local diagram using text and image embeddings in one vector space.
# ==============================================================================

image_embedding_model_id = "cohere.embed-v4.0"
image_paths_setting = "sample_design_diagram.png,sample_sequence_diagram.png"

diagram_paths = []
for item in image_paths_setting.split(","):
    if item.strip():
        candidate = Path(item.strip()).expanduser()
        diagram_paths.append(candidate if candidate.is_absolute() else project_root / candidate)
if not diagram_paths or not all(item.is_file() for item in diagram_paths):
    raise FileNotFoundError("Set image_paths_setting to at least one existing local diagram.")
image_embeddings = OCIGenAIEmbeddings(
    model_id=image_embedding_model_id, **oci_settings
)
image_vectors = np.asarray(image_embeddings.embed_image_batch(
    [str(item) for item in diagram_paths]
))
diagram_query = np.asarray(image_embeddings.embed_query(
    "A citizen service portal connected to backend services and a database"
))
image_denominators = np.linalg.norm(image_vectors, axis=1) * np.linalg.norm(diagram_query)
if np.any(image_denominators == 0):
    raise ValueError("Cannot rank zero-length image or query vectors.")
image_scores = image_vectors @ diagram_query / image_denominators
best_diagram = int(np.argmax(image_scores))
print(f"Best matching diagram: {diagram_paths[best_diagram].name}")
print(f"Cosine similarity: {image_scores[best_diagram]:.3f}")


Best matching diagram: sample_design_diagram.png
Cosine similarity: 0.451


## Running the cookbook

Run cells in order; check model availability before optional sections. See [readme.md](readme.md) for validation and publishing details.
